In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import deque
import gymnasium as gym
from typing import Optional


In [2]:
DEVICE = 'cuda' if torch.cuda.is_available() \
	else 'mps' if torch.mps.is_available() \
	else 'cpu'

print("Dispositivo disponible:",  DEVICE)


Dispositivo disponible: cuda


In [3]:
class LinearQ(nn.Module):
	def __init__(self, state_dim: int):
		super().__init__()
		self.linear = nn.Linear(state_dim, 1)

	def forward(self, state):
		return self.linear(state)


In [4]:
class ActionTreeNode:
	def __init__(self, state_dim: int, depth: int = 0, max_depth: int = 5, buffer_size: int = 3000):
		self.state_dim = state_dim
		self.depth = depth
		self.max_depth = max_depth

		self.is_leaf = True
		self.model = LinearQ(state_dim).to(DEVICE)
		self.buffer = deque(maxlen=buffer_size)
		# self.buffer = []

		self.split_feature: int
		self.split_threshold: float
		self.left: ActionTreeNode
		self.right: ActionTreeNode
	
	def route(self, state):
		if self.is_leaf:
			return self
		if state[self.split_feature] < self.split_threshold:
			return self.left.route(state)
		else:
			return self.right.route(state)


In [ ]:
class LMUT:
	def __init__(self, state_dim: int, n_actions: int, max_depth: int = 5, lr: float = 1e-3, q_mean: float = 0.0, q_std: float = 1.0):
		self.state_dim = state_dim
		self.n_actions = n_actions
		self.max_depth = max_depth
		self.lr = lr
		self.q_mean = q_mean
		self.q_std = q_std

		self.trees: list[ActionTreeNode] = [
			ActionTreeNode(state_dim, max_depth=max_depth)
			for _ in range(n_actions)
		]

		self.feature_influence = np.zeros(state_dim)

	def add_split_influence(self, feature: int, inf_value: float):
		self.feature_influence[feature] += inf_value

	def predict_all_actions(self, state: np.ndarray):
		state_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
		
		q_vals = []
		for a in range(self.n_actions):
			leaf = self.trees[a].route(state)
			with torch.no_grad():
				q = leaf.model(state_t).item() * self.q_std + self.q_mean
			q_vals.append(q)
		return q_vals

	def add_transition(self, state, action: int, teacher_q: float):
		scaled_q = (teacher_q - self.q_mean) / self.q_std
		tree = self.trees[action]
		leaf = tree.route(state)

		leaf.buffer.append((
			state, 
			scaled_q
		))

	def train_leaf(self, leaf: ActionTreeNode, batch_size: int = 64):
		# print(len(leaf.buffer))
		if len(leaf.buffer) < batch_size:
			# print(":(")
			return float('inf')
		# print(":)")

		batch = random.sample(leaf.buffer, batch_size)

		states = torch.tensor(np.array([s for s, _ in batch]), dtype=torch.float32, device=DEVICE)
		targets = torch.tensor(np.array([q for _, q in batch]), dtype=torch.float32, device=DEVICE).unsqueeze(1)

		if not hasattr(leaf, 'optimizer'):
			leaf.optimizer = optim.Adam(leaf.model.parameters(), lr=self.lr)
		criterion = nn.MSELoss()

		preds = leaf.model(states)
		loss = criterion(preds, targets)

		leaf.optimizer.zero_grad()
		loss.backward()
		leaf.optimizer.step()
		return loss.item()
	
	def _compute_variance(self, leaf: ActionTreeNode):
		if len(leaf.buffer) <= 0:
			return 0.0
		q_vals = [q for _, q in leaf.buffer]
		return np.var(q_vals)

	def _try_split(self, node: ActionTreeNode, min_improvement: float = 0.2, batch_size: int = 128):
		# if node.depth >= node.max_depth:
			# return False
		
		if len(node.buffer) < batch_size:
			return False

		states = np.array([s for s, _ in node.buffer])
		q_vals = np.array([q for _, q in node.buffer])

		parent_var = np.var(q_vals)

		best_gain = -1.0
		best_feature = None
		best_thresh = None
		best_left_mask = None
		best_right_mask = None

		n_features = self.state_dim
		for feat in range(n_features):
			feat_vals = states[:, feat]

			thresholds = np.percentile(feat_vals, [25, 50, 75])

			for thresh in thresholds:
				left_mask = feat_vals < thresh
				right_mask = feat_vals >= thresh

				if left_mask.sum() <= 0 or right_mask.sum() <= 0:
					continue
				
				left_var = np.var(q_vals[left_mask])
				right_var = np.var(q_vals[right_mask])

				w_left = left_mask.sum() / len(q_vals)
				w_right = right_mask.sum() / len(q_vals)

				split_var = w_left * left_var + w_right * right_var
				gain = parent_var - split_var

				if gain > best_gain:
					best_gain = gain
					best_feature = feat
					best_thresh = thresh
					best_left_mask = left_mask
					best_right_mask = right_mask

		# print(best_gain)
		if best_gain > min_improvement:
			# Obtener los pesos del modelo
			w = node.model.linear.weight.detach().cpu().numpy().flatten()
			sum_w_sq = np.sum(w**2) + 1e-8
			# Término izquierda
			weight_factor = 1.0 + (w[best_feature]**2) / sum_w_sq

			# Término derecha
			left_q = q_vals[best_left_mask]
			right_q = q_vals[best_right_mask]
			left_var = np.var(left_q)
			right_var = np.var(right_q)
			n_left = len(left_q)
			n_right = len(right_q)
			n_total = len(q_vals)

			weighted_child_var = (n_left * left_var + n_right * right_var) / n_total
			var_reduction = parent_var - weighted_child_var

			influence = weight_factor * var_reduction
			self.add_split_influence(best_feature, influence)

			node.is_leaf = False
			node.split_feature = best_feature
			node.split_threshold = best_thresh

			node.left = ActionTreeNode(node.state_dim, node.depth + 1, node.max_depth)
			node.right = ActionTreeNode(node.state_dim, node.depth + 1, node.max_depth)

			node.left.model.load_state_dict(node.model.state_dict())
			node.right.model.load_state_dict(node.model.state_dict())

			for state, q_val in node.buffer:
				if state[best_feature] < best_thresh:
					node.left.buffer.append((state, q_val))
				else:
					node.right.buffer.append((state, q_val))
			
			node.buffer.clear()

			self.train_leaf(node.left)
			self.train_leaf(node.right)

			return True
		return False

	def update_all_leaves(self):
		def traverse(node: ActionTreeNode):
			if node.is_leaf:
				loss = self.train_leaf(node)
				print(f"Matt: {loss}")
				# if loss <= 0.05:
				split_occurred = self._try_split(node)
				# else:
					# split_occurred = False

				if loss != float('inf'):
					return 1, loss, 1, 1 if split_occurred else 0
				else:
					return 1, 0.0, 0, 1 if split_occurred else 0
				
			else:
				left_lc, left_loss, left_trained, left_splits = traverse(node.left)
				right_lc, right_loss, right_trained, right_splits = traverse(node.right)
				return (left_lc + right_lc, 
						left_loss + right_loss, 
						left_trained + right_trained, 
						left_splits + right_splits)

		total_leaves = 0
		total_loss = 0
		total_trained = 0
		total_splits = 0
		for tree in self.trees:
			lc, loss_sum, trained, splits = traverse(tree)
			total_leaves += lc
			total_loss += loss_sum
			total_trained += trained
			total_splits += splits

		avg_loss = total_loss / total_trained if total_trained > 0 else 0.0
		return {'leaf_count': total_leaves, 'avg_loss': avg_loss, 'splits': total_splits}
	
	def print_tree(self, action: int, node: Optional[ActionTreeNode] = None, indent: str = ""):
		if node is None:
			print(f"\n=== Tree for action {action} ===")
			node = self.trees[action]
		
		if node.is_leaf:
			weights = node.model.linear.weight.detach().cpu().numpy().flatten()
			bias = node.model.linear.bias.detach().cpu().item()
			print(f"{indent}[Leaf] depth={node.depth} | y = {weights} * s + {bias:.4f}")
		else:
			print(f"{indent}[Node] depth={node.depth} | split: feature {node.split_feature} < {node.split_threshold:.4f}")
			print(f"{indent}  left:")
			self.print_tree(action, node.left, indent + "    ")
			print(f"{indent}  right:")
			self.print_tree(action, node.right, indent + "    ")

	def compute_current_mse(self):
		total_se = 0.0
		total_n = 0
		for tree in self.trees:
			stack = [tree]
			leaves: list[ActionTreeNode] = []
			while stack:
				node = stack.pop()
				if node.is_leaf:
					leaves.append(node)
				else:
					stack.append(node.left)
					stack.append(node.right)
			for leaf in leaves:
				for state, scaled_q in leaf.buffer:
					state_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
					with torch.no_grad():
						pred_scaled = leaf.model(state_t).item()
					pred_orig = pred_scaled * self.q_std + self.q_mean
					target_orig = scaled_q * self.q_std + self.q_mean
					total_se += (pred_orig - target_orig) ** 2
					total_n += 1
		return total_se / total_n if total_n > 0 else 0.0
	
	def _serialize_node(self, node: ActionTreeNode):
		data = {
			'is_leaf': node.is_leaf,
			'depth': node.depth,
			'max_depth': node.max_depth,
			'state_dim': node.state_dim,
		}
		if node.is_leaf:
			data['model_state'] = node.model.state_dict()
		else:
			data['split_feature'] = int(node.split_feature)
			data['split_threshold'] = float(node.split_threshold)
			data['left'] = self._serialize_node(node.left)
			data['right'] = self._serialize_node(node.right)
		return data

	def _deserialize_node(self, data: dict):
		node = ActionTreeNode(
			state_dim=data['state_dim'],
			depth=data['depth'],
			max_depth=data['max_depth'],
		)
		node.is_leaf = data['is_leaf']
		if node.is_leaf:
			node.model.load_state_dict(data['model_state'])
			node.model.to(DEVICE)
		else:
			node.split_feature = data['split_feature']
			node.split_threshold = data['split_threshold']
			node.left = self._deserialize_node(data['left'])
			node.right = self._deserialize_node(data['right'])
		return node

	def save(self, path: str):
		checkpoint = {
			'config': {
				'state_dim': self.state_dim,
				'n_actions': self.n_actions,
				'max_depth': self.max_depth,
				'lr': self.lr,
				'q_mean': self.q_mean,
				'q_std': self.q_std,
			},
			'feature_influence': self.feature_influence.tolist(),
			'trees': [self._serialize_node(tree) for tree in self.trees],
		}
		torch.save(checkpoint, path)

	@classmethod
	def load(cls, path: str):
		checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)

		cfg = checkpoint['config']
		model = cls(
			state_dim=cfg['state_dim'],
			n_actions=cfg['n_actions'],
			max_depth=cfg['max_depth'],
			lr=cfg['lr'],
			q_mean=cfg['q_mean'], 
			q_std=cfg['q_std'],		
			)
		model.feature_influence = np.array(checkpoint['feature_influence'])
		model.trees = [
			model._deserialize_node(tree_data)
			for tree_data in checkpoint['trees']
		]
		return model

In [6]:
env = gym.make('CartPole-v1')
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

print(state_dim, n_actions)


4 2


In [7]:
class DQNNetwork(nn.Module):
	"""
	Red neuronal feedforward para aproximar Q-values.
	
	Arquitectura:
	- Capa de entrada: recibe el estado del entorno
	- 2 capas ocultas Linear con activación ReLU
	- Capa de salida Linear: devuelve Q-value para cada acción posible
	
	Parámetros
	----------
	state_size : int
		Dimensión del espacio de estados (número de features de observación)
	action_size : int
		Número de acciones posibles
	hidden_size : int, opcional
		Número de neuronas en las capas ocultas (default: 128)
	"""
	
	def __init__(self, state_size: int, action_size: int, hidden_size: int = 128):
		super(DQNNetwork, self).__init__()

		# Capa de tipo secuencial que contiene las capas de la red
		# Lineal + ReLU + Lineal + ReLU + Lineal
		self.seq = nn.Sequential(
			nn.Linear(state_size, hidden_size),
			nn.ReLU(),
			nn.Linear(hidden_size, hidden_size),
			nn.ReLU(),
			nn.Linear(hidden_size, action_size)
		)
		
	def forward(self, state: torch.Tensor) -> torch.Tensor:
		"""
		Forward pass de la red.
		
		Parámetros
		----------
		state : torch.Tensor
			Estado(s) del entorno. Shape: (batch_size, state_size) o (state_size,)
			
		Retorna
		-------
		torch.Tensor
			Q-values para cada acción. Shape: (batch_size, action_size) o (action_size,)
		"""
		# Aplica la red neuronal al estado de entrada y devuelve el resultado
		return self.seq(state)


In [8]:
def epsilon_greedy_policy(policy_net: DQNNetwork, state: np.ndarray, epsilon: float, action_size: int) -> int:
	"""
	Selecciona una acción usando epsilon-greedy policy.
	
	Con probabilidad epsilon elige una acción aleatoria (exploración), y con probabilidad 
	1-epsilon elige la  mejor acción según la red (explotación).
	
	Parámetros
	----------
	policy_net : DQNNetwork
		Red neuronal que representa la política del agente
	state : np.ndarray
		Estado actual del entorno como array
	epsilon : float
		Probabilidad de elegir una acción aleatoria
	action_size : int
		Número de acciones disponibles en el entorno
		
	Retorna
	-------
	int
		Acción seleccionada
	"""
	# Exploración: acción aleatoria
	if np.random.random() < epsilon:
		# Devolver acción aleatoria con np.random.randint
		return np.random.randint(action_size)
	
	# Explotación: mejor acción según Q-values
	with torch.no_grad():
		# Transformar el estado a un tensor
		state = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)

		# aplicar la red al estado para obtener los valores q
		q_values = policy_net.forward(state)
		
		# devolver el índice del mayor valor Q (como int)
		return q_values.argmax().item()
	

In [9]:
def play_episode(env: gym.Env, policy_net: DQNNetwork):
	"""
	Simula un episodio usando una política e-greedy.

	Parámetros:
		env: entorno de gym.
		policy_net: red que define la política del agente.

	Devuelve:
		Tupla (recompensa_total, pasos_totales) .
	"""
	state, info = env.reset()
	episode_steps = episode_reward = 0
	done = False
	while not done:
		# Selecciona la acción adecuada usando la función epsilon_greedy_policy
		# IMPORTANTE: epsilon tiene que ser 0 para que no haga acciones aleatorias
		action = epsilon_greedy_policy(policy_net,state, 0, env.action_space.n)

		next_state, reward, terminated, truncated, info = env.step(action)
		
		episode_steps += 1
		episode_reward += reward
		done = terminated or truncated
		state = next_state
	return episode_reward, episode_steps


In [10]:
final_model_path = 'models/dqn_cartpole'

policy_net = torch.load(final_model_path, weights_only=False)
policy_net.eval()


DQNNetwork(
  (seq): Sequential(
    (0): Linear(in_features=4, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [11]:
episode_reward, episode_steps = play_episode(env, policy_net)
print(f"Recompensa del episodio: {episode_reward}")
print(f"Pasos del episodio: {episode_steps}")


Recompensa del episodio: 500.0
Pasos del episodio: 500


In [ ]:
def active_play_mimic(env, teacher, mimic_lmut, episodes=200, collect_steps=2000, train_iters=50, epsilon_start=1.0, epsilon_end=0.05, decay=0.995):
	epsilon = epsilon_start
	buffer = []

	for ep in range(episodes):
		state, _ = env.reset()
		done = False
		episode_reward = 0
		while not done:
			action = epsilon_greedy_policy(teacher, state, epsilon, env.action_space.n)
			state_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
			with torch.no_grad():
				teacher_q = teacher.forward(state_t)[0, action].item()
			buffer.append((state, action, teacher_q))

			next_state, reward, terminated, truncated, _ = env.step(action)
			episode_reward += reward
			done = terminated or truncated
			state = next_state

		# entrena por lotes grandes, no a cada paso
		if len(buffer) >= collect_steps:
			for s, a, tq in buffer:
				mimic_lmut.add_transition(s, a, tq)
			for _ in range(train_iters):
				stats = mimic_lmut.update_all_leaves()
			mse = mimic_lmut.compute_current_mse()
			print(f"Ep {ep+1} | Leaves: {stats['leaf_count']} | "
				  f"MSE: {mse:.4f} | Splits acum: {stats['splits']} | ε: {epsilon:.3f}")
			buffer.clear()

		epsilon = max(epsilon_end, epsilon * decay)

In [ ]:
def active_play_mimic(env, teacher, mimic_lmut, episodes=200,
                      buffer_size=3000, train_every=1000, train_iters=1,
                      epsilon_start=1.0, epsilon_end=0.1, decay=0.995):
    from collections import deque
    epsilon = epsilon_start
    buffer = deque(maxlen=buffer_size)
    steps_since_train = 0

    for ep in range(episodes):
        state, _ = env.reset()
        done = False
        while not done:
            action = epsilon_greedy_policy(teacher, state, epsilon, env.action_space.n)
            state_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            with torch.no_grad():
                teacher_q = teacher.forward(state_t)[0, action].item()
            buffer.append((state, action, teacher_q))
            steps_since_train += 1

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            state = next_state

            if steps_since_train >= train_every and len(buffer) >= train_every:
                for s, a, tq in buffer:
                    mimic_lmut.add_transition(s, a, tq)
                for _ in range(train_iters):
                    stats = mimic_lmut.update_all_leaves()
                mse = mimic_lmut.compute_current_mse()

                print(
					f"Episode {ep+1} | "
					f"Leaves: {stats['leaf_count']} | "
					f"AvgLoss: {stats['avg_loss']:.4f} | "
					f"MSE: {mse:.4f} | "
					f"Splits: {stats['splits']} | "
					f"ε: {epsilon:.3f}"
				)
                steps_since_train = 0
                buffer.clear()

        epsilon = max(epsilon_end, epsilon * decay)

In [ ]:
mimic = LMUT(state_dim, n_actions, max_depth=5, q_mean=424.0931, q_std=45.0807)
# mimic = LMUT.load("mimic")
active_play_mimic(env, policy_net, mimic, episodes=200)


In [15]:
mimic.print_tree(0)



=== Tree for action 0 ===
[Node] depth=0 | split: feature 3 < 0.3674
  left:
    [Node] depth=1 | split: feature 2 < -0.0689
      left:
        [Node] depth=2 | split: feature 3 < -1.0423
          left:
            [Node] depth=3 | split: feature 2 < -0.1606
              left:
                [Leaf] depth=4 | y = [-0.7279062  -0.5872766   0.28661108 -0.05161154] * s + -0.2453
              right:
                [Leaf] depth=4 | y = [-0.9551516  -0.9081525   0.62540346  0.28540748] * s + -0.5817
          right:
            [Node] depth=3 | split: feature 2 < -0.1415
              left:
                [Node] depth=4 | split: feature 3 < -0.3447
                  left:
                    [Node] depth=5 | split: feature 2 < -0.1753
                      left:
                        [Leaf] depth=6 | y = [-1.018096   -0.8163887   0.68139184  0.34208754] * s + -0.6359
                      right:
                        [Leaf] depth=6 | y = [-0.7011822  -0.46533635  0.68881005  0.350

In [16]:
def evaluate_fidelity(env, teacher: DQNNetwork, mimic_lmut: LMUT, num_samples: int = 1000):
	mae_sum = 0.0
	mse_sum = 0.0
	count = 0
	correct = 0

	state, _ = env.reset()
	for _ in range(num_samples):
		state_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
		with torch.no_grad():
			q_all = teacher.forward(state_t)
			# print(q_all)
			teacher_action = torch.argmax(q_all[0]).item()
			teacher_q = q_all[0, teacher_action].item()

		mimic_qs = mimic_lmut.predict_all_actions(state)
		mimic_q = mimic_qs[teacher_action]
		
		mimic_action = np.argmax(mimic_qs)

		if teacher_action == mimic_action:
			correct += 1

		error = teacher_q - mimic_q
		mae_sum += abs(error)
		mse_sum += error ** 2
		count += 1

		next_state, _, terminated, truncated, _ = env.step(teacher_action)
		if terminated or truncated:
			state, _ = env.reset()
		else:
			state = next_state

	mae = mae_sum / count
	rmse = (mse_sum / count) ** 0.5
	accuracy = correct / count
	return mae, rmse, accuracy

In [17]:
mae, rmse, acc = evaluate_fidelity(env, policy_net, mimic, num_samples=10000)
print(f"Fidelity: MAE = {mae:.4f}, RMSE = {rmse:.4f}, Action Accuracy = {acc:.2%}")


Fidelity: MAE = 18.5570, RMSE = 25.0358, Action Accuracy = 56.21%


In [18]:
def evaluate(env, agent: LMUT, episodes: int = 10):
	total_reward = 0
	for _ in range(episodes):
		state, _ = env.reset()
		done = False
		while not done:
			q_vals = agent.predict_all_actions(np.asarray(state).flatten())
			action = np.argmax(q_vals)
			state, reward, terminated, truncated, _ = env.step(action)
			done = terminated or truncated
			total_reward += reward

	return total_reward / episodes


In [19]:
reward = evaluate(env, mimic, 100)
print("Mimic average return:", reward)


Mimic average return: 186.82


In [20]:
print("\n=== Feature Influence ===")
feature_names = ["Cart Position", "Cart Velocity", "Pole Angle", "Pole Angular Velocity"]
for i, inf in enumerate(mimic.feature_influence):
	print(f"{feature_names[i]:20} : {inf:.6f}")
	


=== Feature Influence ===
Cart Position        : 7.637571
Cart Velocity        : 16.678162
Pole Angle           : 18.480509
Pole Angular Velocity : 34.882773


In [21]:
mimic.save("mimic_2")

In [22]:
mimic.print_tree(0)


=== Tree for action 0 ===
[Node] depth=0 | split: feature 3 < 0.3674
  left:
    [Node] depth=1 | split: feature 2 < -0.0689
      left:
        [Node] depth=2 | split: feature 3 < -1.0423
          left:
            [Node] depth=3 | split: feature 2 < -0.1606
              left:
                [Leaf] depth=4 | y = [-0.7279062  -0.5872766   0.28661108 -0.05161154] * s + -0.2453
              right:
                [Leaf] depth=4 | y = [-0.9551516  -0.9081525   0.62540346  0.28540748] * s + -0.5817
          right:
            [Node] depth=3 | split: feature 2 < -0.1415
              left:
                [Node] depth=4 | split: feature 3 < -0.3447
                  left:
                    [Node] depth=5 | split: feature 2 < -0.1753
                      left:
                        [Leaf] depth=6 | y = [-1.018096   -0.8163887   0.68139184  0.34208754] * s + -0.6359
                      right:
                        [Leaf] depth=6 | y = [-0.7011822  -0.46533635  0.68881005  0.350

In [23]:
hola = LMUT.load("mimic_2")
hola.print_tree(0)


=== Tree for action 0 ===
[Node] depth=0 | split: feature 3 < 0.3674
  left:
    [Node] depth=1 | split: feature 2 < -0.0689
      left:
        [Node] depth=2 | split: feature 3 < -1.0423
          left:
            [Node] depth=3 | split: feature 2 < -0.1606
              left:
                [Leaf] depth=4 | y = [-0.7279062  -0.5872766   0.28661108 -0.05161154] * s + -0.2453
              right:
                [Leaf] depth=4 | y = [-0.9551516  -0.9081525   0.62540346  0.28540748] * s + -0.5817
          right:
            [Node] depth=3 | split: feature 2 < -0.1415
              left:
                [Node] depth=4 | split: feature 3 < -0.3447
                  left:
                    [Node] depth=5 | split: feature 2 < -0.1753
                      left:
                        [Leaf] depth=6 | y = [-1.018096   -0.8163887   0.68139184  0.34208754] * s + -0.6359
                      right:
                        [Leaf] depth=6 | y = [-0.7011822  -0.46533635  0.68881005  0.350

In [24]:
from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt

env = gym.make("CartPole-v1", render_mode="rgb_array")

frames = []
state, info = env.reset()
episode_reward = 0
episode_steps = 0
done = False

while not done:
    frames.append(env.render())
    q_vals = mimic.predict_all_actions(np.asarray(state).flatten())
    action = int(np.argmax(q_vals))
    state, reward, terminated, truncated, info = env.step(action)
    episode_reward += reward
    episode_steps += 1
    done = terminated or truncated

env.close()
print(f"Reward: {episode_reward:.1f} | Pasos: {episode_steps}")

# Mostrar animación
fig, ax = plt.subplots(figsize=(6, 4))
ax.axis('off')
im = ax.imshow(frames[0])

def update(i):
    im.set_data(frames[i])
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=30, blit=True)
plt.close()
HTML(ani.to_jshtml())

c:\Users\Nitropc\Desktop\UCM\MIA\XRL\.venv\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Reward: 81.0 | Pasos: 81


In [25]:
from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt

env = gym.make("CartPole-v1", render_mode="rgb_array")

frames = []
state, info = env.reset()
episode_reward = 0
episode_steps = 0
done = False

while not done:
    frames.append(env.render())
    action = epsilon_greedy_policy(policy_net, state, 0, env.action_space.n)
    state, reward, terminated, truncated, info = env.step(action)
    episode_reward += reward
    episode_steps += 1
    done = terminated or truncated

env.close()
print(f"Reward: {episode_reward:.1f} | Pasos: {episode_steps}")

# Mostrar animación
fig, ax = plt.subplots(figsize=(6, 4))
ax.axis('off')
im = ax.imshow(frames[0])

def update(i):
    im.set_data(frames[i])
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=30, blit=True)
plt.close()
HTML(ani.to_jshtml())

Reward: 500.0 | Pasos: 500
